# TimeR4 v7 — Rerank thời gian chính xác (biến recall thành Hit@1)

**Phát hiện quyết định từ v6:**
- Retriever đã hết là nút thắt: trần recall@50 **23.7% → 57.6%** (gấp 2.4×).
- Nhưng Hit@1 đứng yên (~16%) → nút thắt **dịch xuống rerank/suy luận**.
- Soi 246 câu "có tên đúng nhưng sai": **96% sai vì SAI NGÀY** — fact đúng
  thực thể nhưng sai timestamp, còn fact-đúng-ngày không lọt top-8 feed cho LLM.

**Nguyên nhân kỹ thuật (bug trong rerank v4–v6):**
`time_filter_score` cũ thưởng `diff > 0` (mọi fact TRƯỚC mốc hỏi) và trả `-100`
khi `diff = 0`. Tức **fact khớp ngày chính xác bị phạt nặng** ở câu "vào/in",
còn fact cũ hơn lại được thưởng → fact 2005 thắng fact 2006 đúng ngày.

**Thay đổi của v7 (CHỈ sửa rerank — giữ nguyên retriever v6 để cô lập biến):**

| | Logic mới (port từ `re_rank_single_result`, alpha=0.8) |
|---|---|
| "**vào / in**" (vào 2006-08-25) | thưởng fact RƠI ĐÚNG khoảng thời gian câu hỏi, phạt nặng phần còn lại |
| "**trước**" | thưởng fact trước mốc, càng gần càng cao; phạt fact ở/sau |
| "**sau**" | thưởng fact sau mốc, càng gần càng cao; phạt fact ở/trước |
| "**đầu tiên / cuối cùng**" | sắp xếp theo thứ tự thời gian |
| Độ chi tiết | khớp theo granularity câu hỏi (năm / tháng / ngày) |

**🔬 Cell chứng minh đòn bẩy:** đo tỉ lệ "fact-đúng-ngày lọt top-8" dưới
rerank CŨ vs MỚI (chỉ tính trên câu mà fact-đúng-ngày CÓ trong top-50) —
chứng minh rerank mới đẩy đúng fact lên, không cần chạy lại LLM.

> ⚠️ Kaggle: GPU T4 x2 · Internet ON · Persistence=Files only · Restart trước Run All


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — CÀI THƯ VIỆN + LOAD MISTRAL
# ═══════════════════════════════════════════════════════════

import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
    'peft', 'sentence-transformers', 'transformers',
    'huggingface-hub', 'accelerate', 'wandb'],
    capture_output=True)

pkgs = [
    'huggingface_hub==0.23.4', 'peft==0.11.1',
    'transformers==4.44.0', 'sentence-transformers==3.3.1',
    'accelerate==0.34.0', 'faiss-cpu', 'tqdm',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "❌"} {pkg}')

import torch, json, glob, sys, re, random, copy
from tqdm import tqdm
from collections import defaultdict
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
print('✅ Import OK')

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  Không có GPU! Vào Session options → ACCELERATOR → GPU T4')

random.seed(42)

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'
print(f'\nLoading {MODEL_NAME}... (~5-8 phút)')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
vram = torch.cuda.memory_allocated()/1e9
print(f'✅ Mistral loaded! VRAM: {vram:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CLONE TIMER4 + LOAD MULTITQ-VI (TRAIN + TEST) + TKG
# ═══════════════════════════════════════════════════════════

if not os.path.exists('/kaggle/working/TimeR4'):
    os.system('git clone --depth 1 https://github.com/qianxinying/TimeR4.git /kaggle/working/TimeR4')
os.chdir('/kaggle/working/TimeR4')

triple_list = []
with open('datasets/MultiTQ/kg/full.txt', encoding='utf-8') as f:
    for line in f:
        line = line.strip().replace('_', ' ')
        if not line: continue
        parts = line.split('\t') if '\t' in line else line.split()
        if len(parts) >= 4:
            triple_list.append(parts[:4])
print(f'✅ TKG: {len(triple_list):,} triplets')

found_train = glob.glob('/kaggle/input/**/MultiTQ_vi_train.json', recursive=True)
found_test  = glob.glob('/kaggle/input/**/MultiTQ_vi_test.json', recursive=True)

if not found_test:
    raise FileNotFoundError('MultiTQ_vi_test.json — cần upload')
with open(found_test[0], encoding='utf-8') as f:
    test_questions = json.load(f)
print(f'✅ Test: {found_test[0]} — {len(test_questions):,} câu')

if found_train:
    with open(found_train[0], encoding='utf-8') as f:
        train_questions = json.load(f)
    print(f'✅ Train: {found_train[0]} — {len(train_questions):,} câu')
else:
    train_questions = None
    print('⚠️  Không có train.json')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — *** FIX #3 + DIRECTIONALITY *** CONSTRUCT 3 LOẠI NEGATIVE
# time / content / both incorrect (Section 4.4)
# v6: khớp đáp án với CẢ subject (triple[0]) LẪN object (triple[2])
# ═══════════════════════════════════════════════════════════

def triple_to_text(triple):
    """Format khớp pipeline thật: '{S} {R} {O} in {T}.'"""
    s, r, o, t = triple[0], triple[1], triple[2], triple[3]
    return f'{s} {r} {o} in {t}.'

def parse_year(t):
    m = re.match(r'(\d{4})', t)
    return int(m.group(1)) if m else None

# Index toàn bộ TKG theo nhiều chiều để có đủ data tạo 3 loại negative
all_subjects  = list({t[0] for t in triple_list})
all_relations = list({t[1] for t in triple_list})
all_objects   = list({t[2] for t in triple_list})
all_times     = list({t[3] for t in triple_list})

sr_index = defaultdict(list)
for triple in triple_list:
    s, r, o, t = triple
    sr_index[(s, r)].append((o, t))

sr_multi_time = {
    k: v for k, v in sr_index.items()
    if len({parse_year(t) for _, t in v if parse_year(t)}) >= 2
}
print(f'Nhóm (subject, relation) có nhiều thời gian: {len(sr_multi_time):,}')


def find_positive(question_item, triple_list):
    """
    v6 — DIRECTIONALITY FIX.
    Tìm fact positive mà gold answer khớp với object (triple[2]) HOẶC subject (triple[0]).
    Trả về (triple, position) với position ∈ {'object', 'subject'}.
    """
    answers = question_item.get('answers', question_item.get('answer', []))
    if isinstance(answers, str): answers = [answers]
    ans_lower = {str(a).lower() for a in answers}

    obj_pos  = [(t, 'object')  for t in triple_list if str(t[2]).lower() in ans_lower]
    subj_pos = [(t, 'subject') for t in triple_list if str(t[0]).lower() in ans_lower]
    pool = obj_pos + subj_pos
    return random.choice(pool) if pool else (None, None)


def construct_3_negatives(positive, ans_position, sr_multi_time, triple_list,
                          all_subjects, all_objects, all_times):
    """
    Corrupt 3 cách độc lập (Section 4.4). v6: 'content incorrect' đổi
    ĐÚNG thực thể mang đáp án (subject nếu đáp án là subject, ngược lại object).
    1. Time incorrect:    giữ (s, r, o), đổi timestamp
    2. Content incorrect: giữ timestamp, đổi thực-thể-đáp-án
    3. Both incorrect:    đổi cả thực-thể-đáp-án VÀ timestamp
    """
    s, r, o, t = positive
    pos_year = parse_year(t)
    negatives = {}

    # 1. Time incorrect — cùng (s, r, o), khác thời gian
    candidates_time = sr_multi_time.get((s, r), [])
    time_neg_candidates = [(obj, time) for obj, time in candidates_time
                            if parse_year(time) != pos_year]
    if time_neg_candidates:
        neg_o, neg_t = random.choice(time_neg_candidates)
        negatives['time_incorrect'] = [s, r, o, neg_t]   # giữ nội dung, đổi time

    # Chọn pool & vị trí cần corrupt theo hướng đáp án
    if ans_position == 'subject':
        entity_pool, gold_entity = all_subjects, s
    else:
        entity_pool, gold_entity = all_objects, o

    def _rand_diff(pool, avoid):
        x = random.choice(pool); tries = 0
        while str(x).lower() == str(avoid).lower() and tries < 5:
            x = random.choice(pool); tries += 1
        return x

    # 2. Content incorrect — giữ time, đổi thực-thể-đáp-án
    new_ent = _rand_diff(entity_pool, gold_entity)
    if ans_position == 'subject':
        negatives['content_incorrect'] = [new_ent, r, o, t]
    else:
        negatives['content_incorrect'] = [s, r, new_ent, t]

    # 3. Both incorrect — đổi thực-thể-đáp-án VÀ timestamp
    new_ent2  = _rand_diff(entity_pool, gold_entity)
    new_time2 = random.choice(all_times)
    if ans_position == 'subject':
        negatives['both_incorrect'] = [new_ent2, r, o, new_time2]
    else:
        negatives['both_incorrect'] = [s, r, new_ent2, new_time2]

    return negatives


def build_pairs_v2(questions, sr_multi_time, triple_list,
                    all_subjects, all_objects, all_times, max_pairs, desc='Construct'):
    """Mỗi question tạo TỐI ĐA 3 pairs. v6: theo dõi cả hướng subject/object."""
    pairs = []
    pos_stat = {'object': 0, 'subject': 0}
    for item in tqdm(questions, desc=desc):
        if len(pairs) >= max_pairs: break
        pos, position = find_positive(item, triple_list)
        if not pos: continue
        pos_stat[position] += 1
        negs = construct_3_negatives(pos, position, sr_multi_time, triple_list,
                                     all_subjects, all_objects, all_times)
        q = item.get('question', item.get('Question', ''))
        for neg_type, neg_triple in negs.items():
            pairs.append({
                'question':  q,
                'positive':  triple_to_text(pos),
                'negative':  triple_to_text(neg_triple),
                'neg_type':  neg_type,
                'ans_position': position,
            })
    print(f'   Positive theo hướng → object: {pos_stat["object"]:,} | subject: {pos_stat["subject"]:,}')
    return pairs

print('✅ Hàm construct 3 negative + DIRECTIONALITY (subject/object) sẵn sàng')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — TẠO TRAIN PAIRS + EVAL PAIRS (3 loại negative, không overlap)
# ═══════════════════════════════════════════════════════════

# ── v6 config ──────────────────────────────────────────────
# Retriever ĐA NGÔN NGỮ (thay all-MiniLM-L6-v2 chỉ-tiếng-Anh).
#   Drop-in, không cần prefix: 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
#   Mạnh hơn (cần prefix query:/passage:): 'intfloat/multilingual-e5-base'
RETRIEVER_BASE = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
RETRIEVER_PATH_NEW = '/kaggle/working/models/finetuned_retriever_v6'  # path mới → fine-tune lại

if train_questions is not None and not os.path.exists(RETRIEVER_PATH_NEW):
    MAX_TRAIN_QUESTIONS = 10000   # mỗi câu → tối đa 3 pairs → ~30,000 pairs
    MAX_EVAL_QUESTIONS  = 400     # → ~1,200 eval pairs

    print('=== Tạo TRAIN pairs (từ MultiTQ_vi_train.json, 3 loại negative) ===')
    random.shuffle(train_questions)
    training_pairs = build_pairs_v2(
        train_questions[:MAX_TRAIN_QUESTIONS], sr_multi_time, triple_list,
        all_subjects, all_objects, all_times, max_pairs=30000, desc='Train pairs'
    )
    print(f'✅ Tạo được {len(training_pairs):,} TRAIN pairs')

    # Thống kê phân bố 3 loại negative
    from collections import Counter
    type_counts = Counter(p['neg_type'] for p in training_pairs)
    print(f'   Phân bố: {dict(type_counts)}')

    print('\n=== Tạo EVAL pairs (từ MultiTQ_vi_test.json — KHÔNG overlap) ===')
    random.shuffle(test_questions)
    eval_pairs = build_pairs_v2(
        test_questions[:MAX_EVAL_QUESTIONS], sr_multi_time, triple_list,
        all_subjects, all_objects, all_times, max_pairs=1200, desc='Eval pairs'
    )
    print(f'✅ Tạo được {len(eval_pairs):,} EVAL pairs')

    assert len(training_pairs) >= 1000, 'Quá ít training pairs!'
    with open('/kaggle/working/training_pairs_v4.json', 'w', encoding='utf-8') as f:
        json.dump(training_pairs, f, ensure_ascii=False, indent=2)
    with open('/kaggle/working/eval_pairs_v4.json', 'w', encoding='utf-8') as f:
        json.dump(eval_pairs, f, ensure_ascii=False, indent=2)

    print('\n=== Ví dụ 3 loại negative cho cùng 1 câu hỏi ===')
    sample_q = training_pairs[0]['question']
    for p in training_pairs[:3]:
        if p['question'] == sample_q:
            print(f"[{p['neg_type']}]")
            print(f"  Question: {p['question']}")
            print(f"  Positive: {p['positive']}")
            print(f"  Negative: {p['negative']}")
            print()
else:
    print(f'✅ Đã có retriever tại {RETRIEVER_PATH_NEW} hoặc không có train.json — bỏ qua tạo pairs')
    training_pairs, eval_pairs = [], []

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — ĐÁNH GIÁ BASELINE + FINE-TUNE RETRIEVER (3 loại negative)
# ═══════════════════════════════════════════════════════════

def test_retriever(model, test_cases, batch_size=64):
    questions  = [c['question'] for c in test_cases]
    positives  = [c['positive'] for c in test_cases]
    negatives  = [c['negative'] for c in test_cases]
    q_embs = model.encode(questions, convert_to_tensor=True, batch_size=batch_size, show_progress_bar=False)
    p_embs = model.encode(positives, convert_to_tensor=True, batch_size=batch_size, show_progress_bar=False)
    n_embs = model.encode(negatives, convert_to_tensor=True, batch_size=batch_size, show_progress_bar=False)
    correct = 0
    for i in range(len(test_cases)):
        if util.cos_sim(q_embs[i], p_embs[i]).item() > util.cos_sim(q_embs[i], n_embs[i]).item():
            correct += 1
    return correct / len(test_cases) * 100

if not os.path.exists(RETRIEVER_PATH_NEW) and training_pairs:
    model_old = SentenceTransformer(RETRIEVER_BASE)
    print(f'Đánh giá retriever NỀN ({RETRIEVER_BASE}) trên EVAL pairs...')
    acc_old = test_retriever(model_old, eval_pairs)
    print(f'✅ Retriever GỐC: {acc_old:.1f}%')
    del model_old; torch.cuda.empty_cache()

    retriever_model = SentenceTransformer(RETRIEVER_BASE)
    print(f'✅ Loaded base model (đa ngôn ngữ): {RETRIEVER_BASE}')

    train_examples = [InputExample(texts=[p['question'], p['positive']]) for p in training_pairs]
    print(f'Training examples: {len(train_examples):,}')

    BATCH_SIZE = 32
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
    train_loss = losses.MultipleNegativesRankingLoss(retriever_model)

    EPOCHS = 2
    print(f'\nFine-tune: {EPOCHS} epochs, batch_size={BATCH_SIZE}, steps={len(train_dataloader)*EPOCHS}')
    retriever_model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=EPOCHS,
        warmup_steps=int(len(train_dataloader) * 0.1),
        show_progress_bar=True,
        output_path=RETRIEVER_PATH_NEW,
    )
    print(f'✅ Fine-tune xong! Lưu tại {RETRIEVER_PATH_NEW}')

    model_new = SentenceTransformer(RETRIEVER_PATH_NEW)
    acc_new = test_retriever(model_new, eval_pairs)
    print(f'\n✅ Retriever FINE-TUNED (3 loại negative): {acc_new:.1f}%')
    print(f'   Cải thiện: {acc_new-acc_old:+.1f}%')
    del model_new; torch.cuda.empty_cache()
else:
    print(f'✅ Bỏ qua — đã có model hoặc không có train data')
    acc_old, acc_new = 0, 0

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — *** FIX #2 *** PATCH retrival.py + BẬT RERANK (Equation 9-10)
# ═══════════════════════════════════════════════════════════

def normalize_temporal_expression(question):
    text = question.lower()
    spans = []
    for m in re.finditer(r'ngày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'ngày\s+(\d{1,2})/(\d{1,2})/(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(2)}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'tháng\s+(\d{1,2})/(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(2)}-{int(m.group(1)):02d}'))
    if not spans: return text
    kept, last_end = [], -1
    for s, e, r in sorted(spans, key=lambda x: -(x[1]-x[0])):
        if s >= last_end:
            kept.append((s, e, r)); last_end = e
    kept.sort(key=lambda x: x[0])
    for s, e, r in reversed(kept):
        text = text[:s] + r + text[e:]
    return text

print('✅ Module TEN loaded')

with open('retrival.py') as f: code = f.read()

patches = [
    ('self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list]',
     'self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list] if id_list is not None else []'),
    ('self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = embedding_size',
     'self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = self.model.get_sentence_embedding_dimension()'),
    ('self.full_time = [datetime.strptime(triple[3], "%Y-%m-%d").date() for triple in triple_list]',
     'self.full_time = []\n        for triple in triple_list:\n            t = triple[3] if len(triple) > 3 else "2000-01-01"\n            if len(t) == 4: t += "-01-01"\n            elif len(t) == 7: t += "-01"\n            try: self.full_time.append(datetime.strptime(t[:10], "%Y-%m-%d").date())\n            except: self.full_time.append(datetime.strptime("2000-01-01", "%Y-%m-%d").date())'),
    ('corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]',
     'if corpus_embeddings.ndim == 1:\n            corpus_embeddings = corpus_embeddings[np.newaxis, :]\n        corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]'),
]
for old, new in patches:
    if old in code: code = code.replace(old, new)
with open('retrival.py', 'w') as f: f.write(code)
if 'retrival' in sys.modules: del sys.modules['retrival']
from retrival import Retrieval
print('✅ retrival.py patched (4 lỗi kỹ thuật)')


# ── FIX #2: Time-filtering function (Equation 9) ──────────
# ── FIX #2 (v7): Rerank thời gian CHÍNH XÁC — port từ re_rank_single_result ──
from datetime import date as _date, timedelta as _td

def extract_timestamp_from_question(question):
    """Sau TEN, câu hỏi chứa timestamp YYYY-MM-DD / YYYY-MM / YYYY."""
    m = re.search(r'(\d{4}(?:-\d{2}(?:-\d{2})?)?)', question)
    return m.group(1) if m else None

# ───────── (GIỮ LẠI hàm CŨ để so sánh trong cell chẩn đoán) ─────────
def time_filter_score_old(tq_str, t_str):
    try:
        def to_date(s):
            s = s[:10]
            if len(s) == 4: s += '-01-01'
            elif len(s) == 7: s += '-01'
            return datetime.strptime(s, '%Y-%m-%d')
        diff = (to_date(tq_str) - to_date(t_str)).days
        return (1 - abs(diff)/3650.0) if diff > 0 else -100
    except Exception:
        return 0

def rerank_facts_old(question, facts, fact_triples_raw=None):
    tq = extract_timestamp_from_question(question)
    if not tq: return facts
    scored = []
    for fact in facts:
        m = re.search(r'in\s+(\d{4}(?:-\d{2}(?:-\d{2})?)?)\.', fact)
        scored.append((fact, time_filter_score_old(tq, m.group(1)) if m else 0))
    valid = [x for x in scored if x[1] != -100]
    pool = valid if valid else scored
    pool.sort(key=lambda x: -x[1])
    return [f for f, s in pool]

# ───────── (MỚI v7) keyword tiếng Việt + chấm điểm theo hướng ─────────
def vi_time_keyword(qnorm):
    ql = qnorm.lower()
    if any(k in ql for k in ['đầu tiên', 'lần đầu', 'sớm nhất']): return 'first'
    if any(k in ql for k in ['cuối cùng', 'gần nhất', 'lần cuối', 'mới nhất']): return 'last'
    if 'trước' in ql: return 'before'
    if 'sau'   in ql: return 'after'
    return 'in'   # vào / ngày / trong / không có → coi như "in"

def q_period(tq):
    """Trả (start, end) bao trùm độ chi tiết của mốc thời gian câu hỏi."""
    tq = tq[:10]
    if len(tq) == 4:
        y = int(tq); return _date(y,1,1), _date(y,12,31)
    if len(tq) == 7:
        y, m = int(tq[:4]), int(tq[5:7])
        nxt = _date(y+1,1,1) if m == 12 else _date(y,m+1,1)
        return _date(y,m,1), nxt - _td(days=1)
    return _date(int(tq[:4]),int(tq[5:7]),int(tq[8:10])), _date(int(tq[:4]),int(tq[5:7]),int(tq[8:10]))

def fact_date(fact):
    m = re.search(r'in\s+(\d{4})-(\d{2})-(\d{2})', fact)
    if m: return _date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    m = re.search(r'in\s+(\d{4})-(\d{2})', fact)
    if m: return _date(int(m.group(1)), int(m.group(2)), 1)
    m = re.search(r'in\s+(\d{4})', fact)
    if m: return _date(int(m.group(1)), 1, 1)
    return None

def time_score_v7(fd, kw, start, end):
    """1.0 = khớp lý tưởng, -100 = sai hướng (phạt nặng)."""
    if fd is None: return 0.0
    if kw == 'in':
        return 1.0 if (start <= fd <= end) else -100.0
    if kw == 'before':
        if fd < start:
            return 1.0 / (1.0 + (start - fd).days / 30.0)   # càng gần mốc càng cao
        return -100.0
    if kw == 'after':
        if fd > end:
            return 1.0 / (1.0 + (fd - end).days / 30.0)
        return -100.0
    return 0.0   # first/last xử lý riêng

ALPHA = 0.8   # giống paper: final = alpha*sim + (1-alpha)*time

def rerank_facts(question, facts, fact_triples_raw=None):
    """v7: fuse similarity-rank với điểm thời gian theo hướng câu hỏi."""
    tq = extract_timestamp_from_question(question)
    if not tq:
        return facts
    kw = vi_time_keyword(question)
    start, end = q_period(tq)
    n = max(len(facts), 1)

    # 'first'/'last' → sắp theo thời gian (facts đã liên quan thực thể)
    if kw in ('first', 'last'):
        dated   = [(f, fact_date(f)) for f in facts]
        with_t  = [x for x in dated if x[1] is not None]
        without = [x for x in dated if x[1] is None]
        with_t.sort(key=lambda x: x[1], reverse=(kw == 'last'))
        return [f for f, _ in with_t] + [f for f, _ in without]

    scored = []
    for pos, f in enumerate(facts):
        sim_proxy = 1.0 - pos / n                       # proxy cho similarity (đã xếp sẵn)
        ts = time_score_v7(fact_date(f), kw, start, end)
        scored.append((f, ALPHA * sim_proxy + (1 - ALPHA) * ts))
    scored.sort(key=lambda x: -x[1])
    return [f for f, _ in scored]

print('✅ Rerank v7 (chính xác theo hướng trước/sau/vào + granularity) sẵn sàng')

def get_q(x): return x.get('question', x.get('Question', ''))

temporal_q = [q for q in test_questions if normalize_temporal_expression(get_q(q)) != get_q(q).lower()]
questions_1000 = temporal_q[:1000]
print(f'\nDùng {len(questions_1000)} câu test (cùng tập với các lần chạy trước)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — *** FIX #1 và #4 *** PROMPT ĐÚNG THEO PAPER
# Figure 5 (Rewrite) và Figure 6 (Reasoning)
# ═══════════════════════════════════════════════════════════

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def call_llm(prompt, max_new_tokens=80):
    inp = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=600).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**inp, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


# ── FIX #4: Rewrite prompt ĐÚNG theo Figure 5 ────────────
def build_rewrite_prompt(fact, question):
    """
    Đúng nguyên văn Figure 5 của paper, chỉ thay ví dụ minh họa
    bằng đúng ví dụ paper dùng (Juan Carlos I / Vietnam).
    """
    return (
        'Replace the temporal fact in questions with explicit timestamps '
        'from the provided facts or your knowledge without any explanation. '
        'If you are not sure about the answer, return the original questions.\n\n'
        'For instance, from the fact:\n'
        '"[Juan Carlos I, Praise or endorse, Vietnam, 2006-02-22]",\n'
        'We can modify the question:\n'
        '"After Vietnam, who was the first to praise Juan Carlos I?"\n'
        'to "After 2006-02-22, who was the first to praise Juan Carlos I?"\n\n'
        'Here is your turn:\n'
        f'Facts: {fact}\n'
        f'Question: {question}'
    )


# ── FIX #1: Reasoning prompt ĐÚNG theo Figure 6 ──────────
def build_reasoning_prompt(facts, question):
    """
    Đúng nguyên văn Figure 6 của paper:
    yêu cầu trả lời đơn giản nhất, dạng LIST tất cả đáp án khả thi.
    """
    return (
        'Based on the historical facts, please answer the given question. '
        'Please keep the answer as simple as possible and return all the '
        'possible answers as a list.\n'
        f'Historical facts: {facts}\n'
        f'Question: {question}'
    )


def gen_answer(prompt, max_new_tokens=64):
    """max_new_tokens=64: đủ cho list nhiều tên mà vẫn ngắn gọn."""
    inp = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1400).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**inp, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return raw.split('\n')[0].strip()


def parse_list_answer(raw_answer):
    """
    LLM được yêu cầu trả lời dạng list ['A', 'B'].
    Parse ra list string thật để so khớp linh hoạt với gold.
    """
    matches = re.findall(r"['\"]([^'\"]+)['\"]", raw_answer)
    if matches:
        return matches
    return [raw_answer.strip('[]\'" .')]


def evaluate_hit(predicted_raw, gold):
    """So khớp: bất kỳ phần tử nào trong predicted list khớp gold."""
    preds = parse_list_answer(predicted_raw)
    golds = gold if isinstance(gold, list) else [gold]
    for pred in preds:
        pred_l = pred.lower().strip()
        for g in golds:
            g_l = str(g).lower().strip()
            if g_l in pred_l or pred_l in g_l:
                return True
    return False

def get_gold(x): return x.get('answer', x.get('answers', x.get('Answer', x.get('answers_simple', '?'))))


# Test nhanh prompt mới
print('=== TEST PROMPT MỚI (đúng theo paper Figure 5 & 6) ===')
test_facts = "['UK hasLeader David Cameron in 2012-05-01.']"
test_q = 'Who led UK in 2012?'
prompt_test = build_reasoning_prompt(test_facts, test_q)
print('--- Reasoning Prompt ---')
print(prompt_test)
ans = gen_answer(prompt_test)
print(f'\nRaw answer: {ans}')
print(f'Parsed: {parse_list_answer(ans)}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — HÀM PIPELINE v5 (4 fix v4 + tăng recall: n & top_k tham số hoá)
# ═══════════════════════════════════════════════════════════

def run_pipeline_v5(qs_raw, triples, retriever_path, use_ten=True, use_rerank=True,
                    n_retrieve=50, top_k_facts=8, label=''):
    # n_retrieve: số fact FAISS lấy về (v4=15 → v5=50)
    # top_k_facts: số fact feed vào LLM reasoning (v4=3 → v5=8)
    qs = copy.deepcopy(qs_raw)
    print(f'\n{"="*60}\n  {label}\n{"="*60}')

    q_list = [get_q(q) for q in qs]
    q_norm = [normalize_temporal_expression(q) for q in q_list] if use_ten else q_list
    if use_ten:
        n = sum(1 for a,b in zip(q_list,q_norm) if a.lower()!=b)
        print(f'[TEN] Normalized: {n}/{len(q_list)} ({n/len(q_list)*100:.1f}%)')

    print('[1/4] Retrieve background (FKS)...')
    r1 = Retrieval(retriever_path, q_norm, triples, None, None)
    d1, c1 = r1.compute_similarity(n=n_retrieve)
    bg = r1.get_result(d1, c1, q_norm, re_rank=False)

    print('[2/4] Rewrite (Figure 5 prompt)...')
    for i in tqdm(range(len(qs)), desc='Rewrite', leave=False):
        q = get_q(qs[i])
        f = (bg[i].get('fact') or [''])[0]
        prompt = build_rewrite_prompt(f, q)
        resp = call_llm(prompt, max_new_tokens=60).split('\n')[0].strip()
        qs[i]['question'] = resp if resp else q

    print('[3/4] Time-aware Retrieve + Rerank (Equation 9-10)...')
    q2 = [get_q(q) for q in qs]
    if use_ten: q2 = [normalize_temporal_expression(q) for q in q2]
    r2 = Retrieval(retriever_path, q2, triples, None, None)
    d2, c2 = r2.compute_similarity(n=n_retrieve)
    fl = r2.get_result(d2, c2, q2, re_rank=False)

    if use_rerank:
        for i in range(len(fl)):
            facts = fl[i].get('fact') or []
            fl[i]['fact'] = rerank_facts(q2[i], facts)

    print('[4/4] Generate (Figure 6 prompt — list format)...')
    results, correct = [], 0
    for i, item in enumerate(tqdm(qs, desc='Generate', leave=False)):
        q     = get_q(item)
        facts = (fl[i].get('fact') or [])[:top_k_facts]
        gold  = get_gold(qs_raw[i])
        prompt = build_reasoning_prompt(facts, q)
        pred  = gen_answer(prompt)
        ok = evaluate_hit(pred, gold)
        if ok: correct += 1
        results.append({
            'question':  get_q(qs_raw[i]),
            'q_norm':    q_norm[i] if use_ten else '',
            'gold':      str(gold),
            'predicted': pred,
            'parsed':    parse_list_answer(pred),
            'correct':   ok,
            'retrieved_facts': facts,   # lưu để soi lỗi retrieve vs LLM
        })

    em = correct / len(results) * 100
    print(f'\n✅ {label}: {correct}/{len(results)} = {em:.2f}%')
    return results, em

print(f'✅ Pipeline v5 ready (mặc định n_retrieve=50, top_k_facts=8)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — 🔬 CHẨN ĐOÁN RECALL@K (tách lỗi retrieve vs lỗi LLM)
# ═══════════════════════════════════════════════════════════
# Ý tưởng: chạy retrieve 1 lần ở n=50, với mỗi câu kiểm tra xem trong
# top-k fact lấy về có fact nào mang ĐÚNG đáp án (object khớp gold) không.
#   - recall@k = tỉ lệ câu có ít nhất 1 fact-đúng trong top-k.
#   - recall@50 chính là TRẦN của pipeline: rerank/LLM không thể vượt qua,
#     vì chúng chỉ chọn lại trong tập đã lấy về (không thêm fact mới).

N_DIAG = 50
print(f'Chạy retrieve chẩn đoán ở n={N_DIAG} trên {len(questions_1000)} câu...')

# Chuẩn hoá câu hỏi giống pipeline (TEN trước retrieve lần 1)
q_diag = [normalize_temporal_expression(get_q(q)) for q in questions_1000]

r_diag = Retrieval(retriever_to_use if 'retriever_to_use' in dir() else
                   (RETRIEVER_PATH_NEW if os.path.exists(RETRIEVER_PATH_NEW) else RETRIEVER_BASE),
                   q_diag, triple_list, None, None)
d_diag, c_diag = r_diag.compute_similarity(n=N_DIAG)
res_diag = r_diag.get_result(d_diag, c_diag, q_diag, re_rank=False)


def gold_objects(item):
    g = get_gold(item)
    if isinstance(g, str):
        # gold đôi khi là chuỗi "['A', 'B']" → tách thô
        inner = re.findall(r"['\"]([^'\"]+)['\"]", g)
        return inner if inner else [g]
    return g if isinstance(g, list) else [g]


def _match(val, golds):
    v = str(val).lower().strip()
    for g in golds:
        gl = str(g).lower().strip()
        if gl and (gl in v or v in gl):
            return True
    return False

def fact_hits_obj(triple, golds):
    """CŨ: chỉ khớp object triple[2]."""
    return len(triple) >= 3 and _match(triple[2], golds)

def fact_hits_either(triple, golds):
    """v6: khớp subject triple[0] HOẶC object triple[2]."""
    return len(triple) >= 3 and (_match(triple[2], golds) or _match(triple[0], golds))


Ks = [3, 5, 8, 10, 20, 50]
recall_obj    = {k: 0 for k in Ks}   # định nghĩa cũ (object-only)
recall_either = {k: 0 for k in Ks}   # định nghĩa v6 (subject-or-object)
n_total = len(questions_1000)

for idx, item in enumerate(questions_1000):
    golds = gold_objects(item)
    triples = res_diag[idx].get('triple', [])   # xếp theo similarity
    rank_obj = next((r for r, tr in enumerate(triples) if fact_hits_obj(tr, golds)), None)
    rank_eit = next((r for r, tr in enumerate(triples) if fact_hits_either(tr, golds)), None)
    for k in Ks:
        if rank_obj is not None and rank_obj < k: recall_obj[k] += 1
        if rank_eit is not None and rank_eit < k: recall_either[k] += 1

recall_at = recall_either   # dùng định nghĩa v6 làm chính

print('\n' + '═'*60)
print('  RECALL@K — fact đúng có trong top-k lấy về không? (ĐO KÉP)')
print('═'*60)
print(f'  {"k":>4} {"object-only (cũ)":>18} {"subj-or-obj (v6)":>18} {"Δ":>6}')
print('  ' + '-'*56)
for k in Ks:
    po = recall_obj[k]    / n_total * 100
    pe = recall_either[k] / n_total * 100
    print(f'  {k:>4} {po:>17.1f}% {pe:>17.1f}% {pe-po:>+5.1f}')
print('═'*60)

ceiling = recall_either[50] / n_total * 100
ceiling_old = recall_obj[50] / n_total * 100
print(f'\n📌 Trần CŨ (object-only)   = {ceiling_old:.1f}%  (chính là 23.7% của v5)')
print(f'📌 Trần MỚI (subj-or-obj) = {ceiling:.1f}%  ← directionality nâng trần lên đây')
print(f'\n📌 TRẦN pipeline (recall@50) = {ceiling:.1f}%')
print(f'   → Dù retrieve/rerank/LLM hoàn hảo, Hit@1 KHÔNG THỂ vượt {ceiling:.1f}%')
print(f'   (các câu còn lại: fact đúng không tồn tại trong TKG hoặc gold là')
print(f'    thực thể vai trò ICEWS khó khớp chuỗi — giới hạn dữ liệu, không phải pipeline)')

r3  = recall_at[3]  / n_total * 100
r8  = recall_at[8]  / n_total * 100
print(f'\n📌 So sánh v4 (top-3) vs v5 (top-8):')
print(f'   recall@3 = {r3:.1f}%   ← v4 chỉ thấy được nhiêu đây')
print(f'   recall@8 = {r8:.1f}%   ← v5 cho LLM thấy thêm  (+{r8-r3:.1f} điểm fact-đúng)')
print(f'   → khoảng {r8-r3:.1f}% câu có fact-đúng mới lộ ra nhờ feed top-8 thay vì top-3')

# lưu để báo cáo
diag = {'recall_object_only':  {str(k): round(recall_obj[k]/n_total*100, 2) for k in Ks},
        'recall_subj_or_obj':  {str(k): round(recall_either[k]/n_total*100, 2) for k in Ks},
        'ceiling_old_obj': round(ceiling_old, 2),
        'ceiling_new_either': round(ceiling, 2),
        'n_total': n_total}
with open('/kaggle/working/recall_diagnostic_v7.json', 'w', encoding='utf-8') as f:
    json.dump(diag, f, ensure_ascii=False, indent=2)
print('\n✅ Đã lưu recall_diagnostic_v5.json')

del r_diag; torch.cuda.empty_cache()


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — 🔬 CHỨNG MINH ĐÒN BẨY RERANK (cũ vs mới)
# Đo: trong câu mà fact-đúng-ngày CÓ trong top-50, rerank có đẩy nó lên top-8 không?
# Dùng lại res_diag (retrieve n=50) từ Cell 9 → KHÔNG tốn thêm LLM.
# ═══════════════════════════════════════════════════════════

def ideal_fact_str(triple_list_facts, golds, start, end):
    """Trả index fact 'lý tưởng' đầu tiên: khớp gold (subj/obj) VÀ rơi đúng khoảng thời gian."""
    for idx, f in enumerate(triple_list_facts):
        fd = fact_date(f)
        if fd is None or not (start <= fd <= end):
            continue
        fl = f.lower()
        if any(str(g).lower() in fl for g in golds):
            return idx
    return None

TOPK = 8
eligible = 0          # số câu có fact-đúng-ngày trong top-50
hit_old  = 0          # rerank CŨ đẩy được vào top-8
hit_new  = 0          # rerank MỚI đẩy được vào top-8

for idx, item in enumerate(questions_1000):
    golds = gold_objects(item)
    qn    = q_diag[idx]
    tq    = extract_timestamp_from_question(qn)
    if not tq:
        continue
    start, end = q_period(tq)
    facts50 = res_diag[idx].get('fact', [])           # ~50 fact, thứ tự similarity
    if not facts50:
        continue
    # có fact-đúng-ngày ở đâu đó trong 50 không?
    if ideal_fact_str(facts50, golds, start, end) is None:
        continue
    eligible += 1
    top_old = rerank_facts_old(qn, facts50)[:TOPK]
    top_new = rerank_facts(qn, facts50)[:TOPK]
    if ideal_fact_str(top_old, golds, start, end) is not None: hit_old += 1
    if ideal_fact_str(top_new, golds, start, end) is not None: hit_new += 1

print('═'*62)
print('  ĐÒN BẨY RERANK — fact-đúng-ngày có lọt top-8 feed cho LLM?')
print('═'*62)
print(f'  Số câu có fact-đúng-ngày trong top-50 (đủ điều kiện): {eligible}')
if eligible:
    print(f'  Rerank CŨ  đẩy vào top-8: {hit_old:>4} ({hit_old/eligible*100:.1f}%)')
    print(f'  Rerank MỚI đẩy vào top-8: {hit_new:>4} ({hit_new/eligible*100:.1f}%)')
    print(f'  → Chênh lệch: {(hit_new-hit_old)/eligible*100:+.1f} điểm fact-đúng-ngày được cứu')
print('═'*62)
print('  (Đây là phần recall sẽ được CHUYỂN HÓA thành Hit@1 nếu LLM đọc đúng.)')

lever = {'eligible': eligible, 'top8_old': hit_old, 'top8_new': hit_new,
         'gain_points': round((hit_new-hit_old)/max(eligible,1)*100, 2)}
with open('/kaggle/working/rerank_lever_v7.json', 'w', encoding='utf-8') as f:
    json.dump(lever, f, ensure_ascii=False, indent=2)
print('✅ rerank_lever_v7.json saved')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — CHẠY PIPELINE v5 (n=50, top_k=8)
# ═══════════════════════════════════════════════════════════
retriever_to_use = RETRIEVER_PATH_NEW if os.path.exists(RETRIEVER_PATH_NEW) else RETRIEVER_BASE
print(f'Dùng retriever: {retriever_to_use}')

# ── Cấu hình thí nghiệm ──────────────────────────────────
# Mặc định chạy 1 lần ở top_k=8 (cân bằng recall vs nhiễu cho Mistral-7B).
# Muốn quét nhiều giá trị để tìm điểm tối ưu, đổi thành: TOP_K_LIST = [3, 5, 8, 10]
# (mỗi giá trị tốn thêm ~1 giờ vì phải gọi LLM lại toàn bộ.)
TOP_K_LIST = [8]
N_RETRIEVE = 50

sweep_results = {}
for tk in TOP_K_LIST:
    res_v5, em_v5 = run_pipeline_v5(
        questions_1000, triple_list, retriever_to_use,
        use_ten=True, use_rerank=True,
        n_retrieve=N_RETRIEVE, top_k_facts=tk,
        label=f'TimeR4 v5 (n={N_RETRIEVE}, top_k={tk})'
    )
    sweep_results[tk] = (res_v5, em_v5)

# chọn cấu hình tốt nhất
best_tk = max(sweep_results, key=lambda k: sweep_results[k][1])
res_v5, em_v5 = sweep_results[best_tk]
print(f'\n🏆 Tốt nhất: top_k={best_tk} → Hit@1 = {em_v5:.2f}%')

with open('/kaggle/working/results_v7.json', 'w', encoding='utf-8') as f:
    json.dump(res_v5, f, ensure_ascii=False, indent=2)
print('✅ Saved results_v7.json')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — BẢNG SO SÁNH + PHÂN RÃ LỖI (retrieve vs LLM)
# ═══════════════════════════════════════════════════════════
print('═'*72)
print('  BẢNG SO SÁNH — BÁO CÁO VỚI THẦY')
print('═'*72)
print(f'  {"Model":<50} {"Hit@1":>8} {"Cải thiện":>10}')
print('  '+'-'*70)
print(f'  {"TimeR4 gốc (paper, tiếng Anh)":<50} {"42.1%":>8} {"—":>10}')
print(f'  {"TimeR4 baseline (VI, chưa TEN)":<50} {"11.80%":>8} {"—":>10}')
print(f'  {"TimeR4 + TEN (v4 trước fix)":<50} {"11.90%":>8} {"+0.10":>10}')
print(f'  {"TimeR4 FULL FIX (v4: n=15, top-3)":<50} {"15.40%":>8} {"+3.50":>10}')
d = em_v5 - 15.40
ds = f'+{d:.2f} ↑' if d > 0 else f'{d:.2f}'
print(f'  {"TimeR4 v5 (n=50, top-8) — tăng recall":<50} {"16.60%":>8} {"+1.20":>10}')
d6 = em_v5 - 16.60
d6s = f"+{d6:.2f} ↑" if d6 > 0 else f"{d6:.2f}"
print(f'  {"TimeR4 v6 (đa ngôn ngữ + directionality)":<50} {"16.20%":>8} {"-0.40":>10}')
d7 = em_v5 - 16.20
d7s = f"+{d7:.2f} ↑" if d7 > 0 else f"{d7:.2f}"
print(f'  {"TimeR4 v7 (rerank thời gian chính xác)":<50} {em_v5:>7.2f}% {d7s:>10}')
print('═'*72)

# ── Phân rã lỗi: trong các câu SAI, bao nhiêu do retrieve, bao nhiêu do LLM ──
wrong = [r for r in res_v5 if not r['correct']]

def has_gold_fact(r):
    """Trong các fact đã feed cho LLM, có fact mang đúng đáp án không?"""
    golds = gold_objects({'answers': r['gold']})
    for f in r.get('retrieved_facts', []):
        m = re.match(r'(.+?)\s+(.+?)\s+(.+?)\s+in\s+', f)  # tách object thô
        fl = f.lower()
        for g in golds:
            gl = str(g).lower().strip()
            if gl and gl in fl:
                return True
    return False

llm_miss = sum(1 for r in wrong if has_gold_fact(r))   # fact đúng CÓ trong ngữ cảnh nhưng LLM trả sai
retr_miss = len(wrong) - llm_miss                      # fact đúng KHÔNG có trong ngữ cảnh

print(f'\n=== PHÂN RÃ {len(wrong)} CÂU SAI ===')
print(f'  Lỗi RETRIEVE (fact đúng không được feed): {retr_miss:>4} ({retr_miss/len(wrong)*100:.1f}%)')
print(f'  Lỗi LLM (có fact đúng nhưng trả lời sai): {llm_miss:>4} ({llm_miss/len(wrong)*100:.1f}%)')
print(f'  → Sau khi tăng recall, nếu lỗi LLM chiếm đa số thì bước tiếp theo')
print(f'    nên tập trung vào prompt/fine-tune LLM, không phải retrieve nữa.')

# ví dụ
print('\n=== 5 VÍ DỤ ===')
for r in res_v5[:5]:
    print(('✅' if r['correct'] else '❌'), 'Q:', r['question'][:55])
    print('   gold:', r['gold'][:45], '| parsed:', r['parsed'])

summary_v5 = {
    'em_v4_fullfix': 15.40,
    'em_v5': 16.60,
    'em_v6': 16.20,
    'em_v7': round(em_v5, 2),
    'improvement_over_v6': round(em_v5 - 16.20, 2),
    'best_top_k': best_tk,
    'n_retrieve': N_RETRIEVE,
    'recall_ceiling_old_obj': diag['ceiling_old_obj'],
    'recall_ceiling_new_either': diag['ceiling_new_either'],
    'n_wrong': len(wrong),
    'wrong_retrieve_miss': retr_miss,
    'wrong_llm_miss': llm_miss,
}
with open('/kaggle/working/summary_v7_FINAL.json', 'w', encoding='utf-8') as f:
    json.dump(summary_v5, f, ensure_ascii=False, indent=2)
print('\n✅ summary_v7_FINAL.json saved → download từ tab Output')
